In [1]:
from pathlib import Path
from time import gmtime, strftime

import boto3
import pandas as pd
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
region = boto3.session.Session().region_name
timestamp = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

prefix = f"sagemaker/tarea06-processing-byoc/{timestamp}"
input_prefix = f"{prefix}/input/raw"
output_prefix = f"{prefix}/output/preprocessed"

raw_s3_uri = f"s3://{bucket}/{input_prefix}/"
processed_s3_uri = f"s3://{bucket}/{output_prefix}/"

print("REPO_ROOT:", REPO_ROOT)
print("raw_s3_uri:", raw_s3_uri)
print("processed_s3_uri:", processed_s3_uri)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
REPO_ROOT: /home/sagemaker-user/Tarea3_ProductoDeDatos
raw_s3_uri: s3://sagemaker-us-east-1-494321812137/sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/input/raw/
processed_s3_uri: s3://sagemaker-us-east-1-494321812137/sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/


In [2]:
#Carga del dataset a S3
s3 = boto3.client("s3")

for filename in ["sales_train.csv", "test.csv"]:
    local_path = REPO_ROOT / "data" / "raw" / filename
    s3.upload_file(str(local_path), bucket, f"{input_prefix}/{filename}")

print("Raw files uploaded.")
!aws s3 ls {raw_s3_uri}

Raw files uploaded.
2026-03-16 01:36:04   94603866 sales_train.csv
2026-03-16 01:36:05    3182735 test.csv


In [3]:
#Preparación de la imagen
account_id = boto3.client("sts").get_caller_identity()["Account"]
ecr_repository = "tarea06-processing-byoc"
image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository}:latest"

%cd {REPO_ROOT}
!aws ecr describe-repositories --repository-names {ecr_repository} || aws ecr create-repository --repository-name {ecr_repository}
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!docker build --network sagemaker -t {ecr_repository} -f processing/container/Dockerfile .
!docker tag {ecr_repository}:latest {image_uri}
!docker push {image_uri}

print("Image URI:", image_uri)

/home/sagemaker-user/Tarea3_ProductoDeDatos
{
    "repositories": [
        {
            "repositoryArn": "arn:aws:ecr:us-east-1:494321812137:repository/tarea06-processing-byoc",
            "registryId": "494321812137",
            "repositoryName": "tarea06-processing-byoc",
            "repositoryUri": "494321812137.dkr.ecr.us-east-1.amazonaws.com/tarea06-processing-byoc",
            "createdAt": "2026-03-15T19:41:09.533000+00:00",
            "imageTagMutability": "MUTABLE",
            "imageScanningConfiguration": {
                "scanOnPush": false
            },
            "encryptionConfiguration": {
                "encryptionType": "AES256"
            }
        }
    ]
}
WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded
DEPRECATED: The legacy builder is deprecated and will b

In [4]:
#carga del job
AttributeErrorprocessor = ScriptProcessor(
    base_job_name="pfs-preprocess-byoc",
    image_uri=image_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    max_runtime_in_seconds=3600,
    sagemaker_session=sagemaker_session,
)

processor.run(
    code=str(REPO_ROOT / "processing" / "preprocess.py"),
    inputs=[
        ProcessingInput(
            source=raw_s3_uri,
            destination="/opt/ml/processing/input/raw",
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=processed_s3_uri,
        )
    ],
    logs=True,
)

INFO:sagemaker:Creating processing-job with name pfs-preprocess-byoc-2026-03-16-01-40-39-021


.......2026-03-16 01:41:42,452 - processing.preprocess - INFO - Iniciando etapa: cargar_datos_raw
2026-03-16 01:41:43,687 - processing.preprocess - INFO - train_raw: filas=2935849 cols=6
2026-03-16 01:41:43,818 - processing.preprocess - INFO - test_raw: filas=214200 cols=3
2026-03-16 01:41:43,819 - processing.preprocess - INFO - Etapa terminada: cargar_datos_raw duracion_seg=1.37
2026-03-16 01:41:43,819 - processing.preprocess - INFO - Iniciando etapa: tipificar_y_filtrar
2026-03-16 01:41:44,474 - processing.preprocess - INFO - train_clean: filas=2935845 cols=6
2026-03-16 01:41:44,613 - processing.preprocess - INFO - test_clean: filas=214200 cols=3
2026-03-16 01:41:44,615 - processing.preprocess - INFO - Etapa terminada: tipificar_y_filtrar duracion_seg=0.80
2026-03-16 01:41:44,615 - processing.preprocess - INFO - Iniciando etapa: construir_panel_con_features
2026-03-16 01:42:00,666 - processing.preprocess - INFO - panel: filas=7497000 cols=55
2026-03-16 01:42:01,030 - processing.prepr

In [6]:
##verificación del output limitado a 10 lineas por el numero de elementos
processed_s3_uri = processed_s3_uri.rstrip("/")
print(processed_s3_uri)

!aws s3 ls {processed_s3_uri}/ --recursive #verificamos la existencia en S3 que 
#los 4 archivos CSV de salida existen en los prefijos correctos.

!aws s3 cp {processed_s3_uri}/train.csv /tmp/train.csv
!aws s3 cp {processed_s3_uri}/valid.csv /tmp/valid.csv
!aws s3 cp {processed_s3_uri}/test_features.csv /tmp/test_features.csv
!aws s3 cp {processed_s3_uri}/test_pairs.csv /tmp/test_pairs.csv

df_train = pd.read_csv("/tmp/train.csv", nrows=10)
df_valid = pd.read_csv("/tmp/valid.csv", nrows=10)
df_test = pd.read_csv("/tmp/test_features.csv", nrows=10)
df_pairs = pd.read_csv("/tmp/test_pairs.csv", nrows=10)

print("train:", df_train.shape)
print("valid:", df_valid.shape)
print("test_features:", df_test.shape)
print("test_pairs:", df_pairs.shape)

df_train.head()

s3://sagemaker-us-east-1-494321812137/sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed
2026-03-16 01:45:14        874 sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/meta.json
2026-03-16 01:45:14   49773027 sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/test_features.csv
2026-03-16 01:45:14    1794442 sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/test_pairs.csv
2026-03-16 01:45:14 1556919430 sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/train.csv
2026-03-16 01:45:14   51003178 sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/valid.csv
download: s3://sagemaker-us-east-1-494321812137/sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/train.csv to ../../../tmp/train.csv
download: s3://sagemaker-us-east-1-494321812137/sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/valid.csv to ../../.

,date_block_num,month,year,month_sin,month_cos,shop_id,item_id,global_mean,global_sum,global_pairs,...,active_1,dead_6,recency,trend_1_3,trend_1_12,ratio_1_12,sold_cum_lag1,log_sales_cum_lag1,never_sold_before,y
0,0,0,0,0.000000,1.000000e+00,2,30,0.000000,0.0,0.0,...,0,1,99,0.0,0.0,0.0,0,0.000000,1,0.0
1,1,1,0,0.500000,8.660254e-01,2,30,2.005251,126780.0,63224.0,...,0,1,99,0.0,0.0,0.0,0,0.000000,1,0.0
2,2,2,0,0.866025,5.000000e-01,2,30,2.033703,121890.0,59935.0,...,0,1,99,0.0,0.0,0.0,0,0.000000,1,1.0
3,3,3,0,1.000000,6.123234e-17,2,30,2.122372,135783.0,63977.0,...,1,0,1,1.0,1.0,1000000.0,1,0.693147,0,0.0
4,4,4,0,0.866025,-5.000000e-01,2,30,1.888155,103165.0,54638.0,...,0,0,2,0.0,0.0,0.0,1,0.693147,0,0.0


In [7]:
job_name = processor.latest_job.job_name
print("job_name:", job_name)

desc = sagemaker_session.sagemaker_client.describe_processing_job(
    ProcessingJobName=job_name
)

print("status:", desc["ProcessingJobStatus"])
print("failure_reason:", desc.get("FailureReason"))

for o in desc["ProcessingOutputConfig"]["Outputs"]:
    print(o["OutputName"], "->", o["S3Output"]["S3Uri"])

job_name: pfs-preprocess-byoc-2026-03-16-01-40-39-021
status: Completed
failure_reason: None
output-1 -> s3://sagemaker-us-east-1-494321812137/sagemaker/tarea06-processing-byoc/2026-03-16-01-35-58/output/preprocessed/
